In [20]:
print("Function loaded:", callable(generate_customer_email_reply))
print("API key found:", bool(OPENROUTER_API_KEY))
print("Model:", MODEL)

Function loaded: True
API key found: True
Model: nex-agi/nex-n2.5-mini:free


In [ ]:
test_reply = generate_customer_email_reply(d
    customer_email="""
Subject: Delayed order

Hello Customer Support,

My order has been delayed for two weeks.
I have contacted your team several times and nobody has replied.
I am very disappointed and need help immediately.

Regards,
Michael
""",
    company_name="Your Company",
    customer_name="Michael",
    customer_emotion="Annoyed or frustrated",
    staff_tone="Warm and empathetic"
)

print(test_reply)

Network error: HTTPSConnectionPool(host='openrouter.ai', port=443): Max retries exceeded with url: /api/v1/chat/completions (Caused by NameResolutionError("HTTPSConnection(host='openrouter.ai', port=443): Failed to resolve 'openrouter.ai' ([Errno 11001] getaddrinfo failed)"))


In [24]:
import os
import requests
import gradio as gr
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = (
    os.getenv("OPENROUTER_API_KEY")
    or os.getenv("OPENROUTER_KEY")
)

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL = "nex-agi/nex-n2.5-mini:free"


def customer_email_responder(
    customer_email,
    company_name,
    customer_name,
    customer_emotion,
    staff_tone
):
    if not customer_email or not customer_email.strip():
        return "Please paste the customer's email first."

    company_name = company_name.strip() or "Our Company"
    customer_name = customer_name.strip() or "Valued Customer"

    system_prompt = f"""
You are a senior customer service email representative for {company_name}.

Read the customer's email and write a professional, warm, natural,
human-sounding email reply.

Customer name: {customer_name}
Customer emotional state: {customer_emotion}
Staff response tone: {staff_tone}

Rules:
- Start with a suitable greeting.
- Acknowledge the customer's actual concern.
- Show empathy when appropriate.
- If the customer is angry or frustrated, remain calm and respectful.
- Give a clear next step.
- Ask for missing information when necessary.
- Use readable professional email language.
- Include a suitable closing.
- Do not argue, blame, or insult the customer.
- Do not invent refunds, policies, delivery dates, order details,
  investigations, escalations, or promises.
- Do not promise a response time.
- Do not mention AI, models, prompts, safety, or classifications.
- Never return "User Safety: safe".
- Return only the email reply.
"""

    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": customer_email
            }
        ],
        "max_tokens": 500,
        "temperature": 0.2
    }

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://jekacode.africa",
        "X-Title": "Customer Email Responder"
    }

    try:
        if not OPENROUTER_API_KEY:
            raise Exception("API key not found")

        response = requests.post(
            OPENROUTER_URL,
            headers=headers,
            json=payload,
            timeout=45
        )

        response.raise_for_status()

        data = response.json()
        reply = data["choices"][0]["message"]["content"].strip()

        if not reply or len(reply) < 25:
            raise Exception("The AI returned an empty or unsuitable reply")

        bad_phrases = [
            "user safety",
            "safety classification",
            "classification:",
            "system prompt",
            "as an ai"
        ]

        if any(item in reply.lower() for item in bad_phrases):
            raise Exception("The AI returned an unsuitable response")

        return reply

    except Exception as error:
        print("API issue:", error)

        return f"""Hello {customer_name},

Thank you for contacting {company_name}.

I’m sorry to hear about the difficulty you have experienced. I understand
how frustrating this situation must be, and I sincerely apologize for the
inconvenience.

Please provide your order number or relevant reference details so our
customer service team can review the matter and provide the appropriate
next step.

Thank you for your patience and understanding.

Kind regards,
Customer Support Team
{company_name}"""


with gr.Blocks(
    title="Customer Email Responder",
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown(
        """
        # Customer Email Responder

        Generate professional, empathetic, tone-aware email replies
        for customer service teams.

        Review and edit every draft before sending.
        """
    )

    with gr.Row():

        with gr.Column():

            company_input = gr.Textbox(
                label="Company name",
                value="Your Company"
            )

            customer_name_input = gr.Textbox(
                label="Customer name",
                placeholder="Enter customer name if known"
            )

            email_input = gr.Textbox(
                label="Incoming customer email",
                placeholder="Paste the customer's email here...",
                lines=12
            )

            emotion_input = gr.Dropdown(
                label="Customer emotional state",
                choices=[
                    "Unknown, assess from the email",
                    "Annoyed or frustrated",
                    "Angry",
                    "Worried or fearful",
                    "Disappointed",
                    "Confused",
                    "Urgent",
                    "Happy and appreciative",
                    "Neutral"
                ],
                value="Unknown, assess from the email"
            )

            tone_input = gr.Dropdown(
                label="Staff response tone",
                choices=[
                    "Warm and empathetic",
                    "Calm and reassuring",
                    "Formal corporate",
                    "Professional and concise",
                    "Friendly and welcoming",
                    "Apologetic and solution-focused",
                    "Firm but respectful",
                    "Premium concierge service"
                ],
                value="Warm and empathetic"
            )

            generate_button = gr.Button(
                "Generate Professional Reply",
                variant="primary"
            )

        with gr.Column():

            reply_output = gr.Textbox(
                label="Review and edit before sending",
                placeholder="Generated email reply will appear here...",
                lines=20,
                interactive=True
            )

            gr.Markdown(
                """
                Review names, order numbers, refunds, policies, dates,
                and promises before sending.
                """
            )

    generate_button.click(
        fn=customer_email_responder,
        inputs=[
            email_input,
            company_input,
            customer_name_input,
            emotion_input,
            tone_input
        ],
        outputs=reply_output
    )


demo.launch()

C:\Users\ELEAZAR GIDEON\AppData\Local\Temp\ipykernel_17024\462432781.py:135: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [23]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Created dataset file at: .gradio\flagged\dataset2.csv
